# Smile Detection — Training

Place class folders directly inside `data/smile-detection/`, or change `dataset_dir`.

In [ ]:
from keras.layers import Input,Conv2D,BatchNormalization,ReLU,MaxPooling2D,GlobalAveragePooling2D,Dense,concatenate,AveragePooling2D
from keras.models import Model,save_model
import glob
import cv2 as cv
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical
import tensorflow as tf


In [ ]:
from pathlib import Path

dataset_dir = Path("data/smile-detection")

In [ ]:
data = []
labels = []

In [ ]:
for item in sorted(dataset_dir.glob("*/*")):
    img = cv.imread(str(item)) if item.is_file() else None
    if img is None:
        continue
    f_img = cv.bilateralFilter(img , d=9 , sigmaColor=75 , sigmaSpace=75)
    std_dev = np.std(f_img)
    print(std_dev)
    if std_dev>=0 and std_dev <= 50 :
        alpha = 1.5
    elif std_dev>50 and std_dev <= 100 :
        alpha = 1.0
    elif std_dev > 100 and std_dev <=150 :
        alpha = 0.75
    elif std_dev > 150 :
        alpha = 0.50
    reso_img = cv.convertScaleAbs(f_img,alpha=alpha,beta=0)
    lab = cv.cvtColor(reso_img,cv.COLOR_RGB2Lab)
    l,a,b = cv.split(lab)
    image = cv.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
    l=image.apply(l)
    lab=cv.merge((l,a,b))
    n_img = cv.cvtColor(lab,cv.COLOR_Lab2RGB)
    r_img = cv.resize(n_img,(25 , 25))
    img = cv.normalize(r_img,None,0,255,cv.NORM_MINMAX)
    data.append(img)
    label = item.parent.name
    labels.append(label)

In [ ]:
data =np.array(data)
le=LabelEncoder()
labels=le.fit_transform(labels)
labels=to_categorical(labels)

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(data,labels,test_size=0.04)

In [ ]:
inputs = Input(shape=(25,25,3))
x=Conv2D(16,(7,7),strides=2,padding='same')(inputs)
x = BatchNormalization()(x)
x=ReLU()(x)
x = MaxPooling2D((3,3),strides=2,padding='same')(x)
for i in range(3) : 
    bn = BatchNormalization()(x)
    relu = ReLU()(bn)
    conv=Conv2D(32,(3,3),padding='same')(relu)
    x = concatenate([x,conv])
bn = BatchNormalization()(x)
relu = ReLU()(bn)
num_filt = int(tf.keras.backend.int_shape(x)[-1]*0.5)
conv=Conv2D(num_filt,(1,1),padding='same')(relu)
x=AveragePooling2D((2,2),strides=2)(conv)
for i in range(3) : 
    bn = BatchNormalization()(x)
    relu = ReLU()(bn)
    conv=Conv2D(64,(3,3),padding='same')(relu)
    x = concatenate([x,conv])
bn = BatchNormalization()(x)
relu = ReLU()(bn)
num_filt = int(tf.keras.backend.int_shape(x)[-1]*0.5)
conv=Conv2D(num_filt,(1,1),padding='same')(relu)
x=AveragePooling2D((2,2),strides=2)(conv)
for i in range(3) : 
    bn = BatchNormalization()(x)
    relu = ReLU()(bn)
    conv=Conv2D(128,(3,3),padding='same')(relu)
    x = concatenate([x,conv])
x= GlobalAveragePooling2D()(x)
outputs = Dense(2,activation='softmax')(x)
model = Model(inputs,outputs)

In [ ]:
model.compile(optimizer="adam" , loss="categorical_crossentropy" , metrics=["accuracy"])

In [ ]:
model.fit(x_train,y_train,batch_size=100,epochs=1 ,validation_data=(x_test,y_test))

In [ ]:
model.save("Smile1_model.keras")